# ⚽ Pipeline completo — Football Tracking → Eventos

Corré las celdas **de arriba hacia abajo** una vez. Hace todo: clona el repo, instala, monta Drive,
detecta tu modelo entrenado, trackea el video, genera los eventos y te los descarga.

> Activá GPU: **Entorno de ejecución → Cambiar tipo de entorno → GPU**.


## 0. GPU


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU — activá una en Entorno de ejecución')


## 1. Clonar repo + instalar

Instala todo y **reinicia el entorno una vez** (necesario en Colab para que las dependencias nuevas
tomen efecto y no rompan los imports). Cuando te lo pida: **Entorno de ejecución → Reiniciar entorno**
y volvé a **Ejecutar todo** — esta celda se saltea sola la segunda vez.


In [ ]:
import os
REPO_DIR = '/content/ncf_event_tracker'
FLAG = '/content/.setup_done'

if not os.path.exists(FLAG):
    if not os.path.exists(REPO_DIR):
        !git clone -q https://github.com/pipachiesa/ncf_event_tracker.git {REPO_DIR}
    !pip install -q -r {REPO_DIR}/requirements.txt
    !pip install -q filterpy scipy
    !pip install -q -U ultralytics
    !pip install -q --force-reinstall --no-cache-dir pillow numpy
    open(FLAG, 'w').close()
    print('\n' + '='*64)
    print('✅ DEPENDENCIAS INSTALADAS.')
    print('👉 Entorno de ejecución → REINICIAR entorno, y volvé a Ejecutar todo.')
    print('='*64)
    raise SystemExit('Reiniciá el entorno y volvé a Ejecutar todo.')

%cd {REPO_DIR}
import ultralytics; print('ultralytics:', ultralytics.__version__)
assert os.path.exists('data_cleanup/main.py'), 'No se clonó el repo correcto'


## 2. Montar Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR    = '/content/drive/MyDrive/football_analytics'
TRACKING_DIR = os.path.join(DRIVE_DIR, 'tracking_output')
EVENTS_DIR   = os.path.join(DRIVE_DIR, 'event_output')
for d in (DRIVE_DIR, TRACKING_DIR, EVENTS_DIR): os.makedirs(d, exist_ok=True)
print('Resultados en:', DRIVE_DIR)


## 3. Detector

Busca tu **modelo entrenado** en Drive automáticamente. Si no hay, usa el community `football`.


In [ ]:
import glob
from ultralytics import YOLO
hits = glob.glob(os.path.join(DRIVE_DIR, 'models', '**', 'best.pt'), recursive=True)
if hits:
    DETECTOR = hits[0]
    print('✅ Modelo entrenado:', DETECTOR)
    print('   carga OK, clases:', YOLO(DETECTOR).names)
else:
    DETECTOR = 'football'
    print('⚠️ No encontré modelo entrenado en Drive — usando community \'football\'')


## 4. Subir el video

Dejá `DRIVE_VIDEO_PATH` vacío para subir desde tu compu, o poné una ruta de Drive.


In [ ]:
DRIVE_VIDEO_PATH = ''
if DRIVE_VIDEO_PATH:
    assert os.path.exists(DRIVE_VIDEO_PATH), DRIVE_VIDEO_PATH
    VIDEO_PATH = DRIVE_VIDEO_PATH
else:
    from google.colab import files
    print('Seleccioná un .mp4...')
    up = files.upload()
    VIDEO_PATH = os.path.abspath(list(up.keys())[0])
VIDEO_NAME = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
print('Video:', VIDEO_PATH)


## 5. Tracking

Detecta jugadores y balón con tu modelo + homografía. **Es el paso más lento.**


In [ ]:
cmd = (
    f'python data_cleanup/main.py '
    f'--video "{VIDEO_PATH}" '
    f'--output "{TRACKING_DIR}" '
    f'--player-model "{DETECTOR}" '
    f'--ball-model "{DETECTOR}" '
    f'--imgsz 1280 '
    f'--pitch-imgsz 1280 '
    f'--ball-conf 0.1 '
    f'--ball-interp-gap 15 '
    f'--track-buffer 150 '
    f'--min-track-frames 12 '
    f'--pitch-model football-field '
    f'--homography-every 5'
)
print(cmd, '\n')
!{cmd}
TRACKING_CSV = os.path.join(TRACKING_DIR, VIDEO_NAME + '.csv')
assert os.path.exists(TRACKING_CSV), 'No se generó el CSV — mirá el error de arriba'
print('✅ Tracking CSV:', TRACKING_CSV)


## 6. Generar eventos


In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'data_cleanup'))
from lib.match import Match
match = Match()
match.import_raw_data(os.path.dirname(TRACKING_CSV) + os.sep, os.path.basename(TRACKING_CSV))
print(f'Importados {match.frames} frames y {len(match.players)} objetos.')
events = match.generate_events()
print('Resumen:', events.summary())
EVENTS_CSV = os.path.join(EVENTS_DIR, VIDEO_NAME + '_events.csv')
events.export(path=EVENTS_DIR + os.sep, file_name=os.path.basename(EVENTS_CSV))
print('✅ Eventos:', EVENTS_CSV)


## 7. Chequeo rápido (fragmentación de jugadores)


In [ ]:
import csv
from collections import Counter
life = Counter()
for r in csv.DictReader(open(TRACKING_CSV)):
    if r['Object'] == 'player': life[r['Object ID']] += 1
print('IDs de jugador distintos:', len(life), ' (menos = mejor; con el modelo chico eran ~184)')


## 8. Visualización


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Circle
df = pd.read_csv(EVENTS_CSV)
print(df['Type'].value_counts())

def draw_pitch(ax):
    ax.add_patch(plt.Rectangle((0,0),1,1,fill=False,color='black',lw=2))
    ax.plot([0.5,0.5],[0,1],color='black',lw=1)
    ax.add_patch(Circle((0.5,0.5),0.083,fill=False,color='black',lw=1))
    ax.add_patch(plt.Rectangle((0,0.21),0.16,0.58,fill=False,color='black',lw=1))
    ax.add_patch(plt.Rectangle((0.84,0.21),0.16,0.58,fill=False,color='black',lw=1))
    ax.set_xlim(-0.05,1.05); ax.set_ylim(-0.05,1.05)   # Y normal (coords reales de campo)
    ax.set_aspect(68.0/105.0); ax.axis('off')

plot_df = df.dropna(subset=['Start X','Start Y'])
types = sorted(plot_df['Type'].unique())
palette = list(plt.cm.tab10.colors)
colors = {t: palette[i % len(palette)] for i,t in enumerate(types)}
fig, ax = plt.subplots(figsize=(12,8))
ax.add_patch(plt.Rectangle((0,0),1,1,color='#3a8a3a',alpha=0.12,zorder=0))
draw_pitch(ax)
for t in types:
    s = plot_df[plot_df['Type']==t]
    ax.scatter(s['Start X'], s['Start Y'], s=120, color=colors[t], edgecolors='black', linewidths=0.6, label=f'{t} ({len(s)})', zorder=3)
ax.legend(loc='upper center', bbox_to_anchor=(0.5,-0.02), ncol=4, frameon=False)
ax.set_title(f'Eventos detectados — {VIDEO_NAME}')
fig_path = os.path.join(EVENTS_DIR, VIDEO_NAME + '_event_map.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight'); plt.show()
print('Mapa:', fig_path)


## 9. Descargar los CSV


In [ ]:
from google.colab import files
files.download(TRACKING_CSV)
files.download(EVENTS_CSV)
